# Activity 3: Snowpark First Flight

**Module:** Week 6 Day 1
**Estimated Time:** 30 to 40 minutes
**Format:** Individual, partner check at the end
**Prerequisites:** The Snowpark code-along, your `snow.cfg` beside this notebook

## Objective

Fly solo with Snowpark. You will build a tiny claims table in your own schema, compute two window results on it, and write an enriched table back. The expected numbers are given at every step, so you will know immediately whether your Snowpark code is correct.

Copy this notebook to `student-work/week6/day1/` and copy your `snow.cfg` beside it.

## Step 0: Create the table (run once in Snowsight)

```sql
USE ROLE DE;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE TECHCATALYST;
USE SCHEMA TECHCATALYST.<YOUR_NAME>;

CREATE OR REPLACE TRANSIENT TABLE W5D1_CLAIMS (
  region      VARCHAR(10),
  claim_month VARCHAR(7),
  total_paid  NUMBER(12, 2)
);

INSERT INTO W5D1_CLAIMS (region, claim_month, total_paid) VALUES
  ('East', '2026-01', 52000), ('East', '2026-02', 61000), ('East', '2026-03', 58000),
  ('East', '2026-04', 61000), ('East', '2026-05', 70000), ('East', '2026-06', 66000),
  ('West', '2026-01', 43000), ('West', '2026-02', 43000), ('West', '2026-03', 55000),
  ('West', '2026-04', 49000), ('West', '2026-05', 62000), ('West', '2026-06', 71000);

SELECT COUNT(*) FROM W5D1_CLAIMS;  -- 12
```

Why a new table instead of the STOCKS ones? Those are read-only, and Step 4 writes a table back. Snowpark needs a schema you own on both ends.

## Step 1: Connect and point at the table

Create a session from `snow.cfg`, point a DataFrame at `W5D1_CLAIMS`, and preview it.

**Checkpoint:** 12 rows, columns `REGION`, `CLAIM_MONTH`, `TOTAL_PAID`.

In [ ]:
from snowflake.snowpark import Session
from configparser import ConfigParser

# YOUR CODE HERE: read snow.cfg, build params, create the session


In [ ]:
# YOUR CODE HERE: point at W5D1_CLAIMS, print the row count, show the rows


## Step 2: Totals per region

In Activity 2 you computed each region's grand total. Reproduce it with `group_by` and `agg`.

**Checkpoint:** East 368000, West 323000.

In [ ]:
from snowflake.snowpark.functions import col, sum as sum_

# YOUR CODE HERE: group by REGION, sum TOTAL_PAID, alias it REGION_TOTAL, show


## Step 3: Month-over-month change per region

Reproduce the Activity 2 `LAG` drill: partition by region, order by month, previous month's total and the change.

**Checkpoint:** East changes are NULL, +9000, -3000, +3000, +9000, -4000. West changes are NULL, 0, +12000, -6000, +13000, +9000.

In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.functions import lag

# YOUR CODE HERE: build the window (partition REGION, order CLAIM_MONTH),
# add PREV_TOTAL and MOM_CHANGE columns, sort by REGION, CLAIM_MONTH, show all 12 rows


## Step 4: Write it back

Save your enriched DataFrame as `W5D1_CLAIMS_ENRICHED` (overwrite mode), then count it through a fresh `session.table` call.

**Checkpoint:** 12 rows.

In [ ]:
# YOUR CODE HERE: save_as_table, then count the new table


## Step 5: Verify in Snowsight

Open a Snowsight worksheet and run:

```sql
SELECT * FROM W5D1_CLAIMS_ENRICHED ORDER BY REGION, CLAIM_MONTH;
```

Confirm the numbers match Step 3, then close your session in the cell below.

In [ ]:
# YOUR CODE HERE: close the session


## Wrap up

Answer in one or two sentences each, in a markdown cell:

1. At which exact line of your notebook did Snowflake first execute a query on your data?
2. You need to load a CSV that is on your laptop into Snowflake. Snowpark session or `write_pandas`? Why?
3. You need the average of a 50 million row table. Why is `to_pandas` first and averaging second the wrong order?